# 🚀 Huấn Luyện Mô Hình YOLOv8n - IoT Smart Sorting System

Notebook này dùng để huấn luyện mô hình nhận diện **3 dòng sản phẩm màu công nghiệp** bằng GPU miễn phí trên **Google Colab**:
- `0: RED` (Sản phẩm màu đỏ)
- `1: YELLOW` (Sản phẩm màu vàng)
- `2: BLUE` (Sản phẩm màu xanh dương)

### ⚙️ Chuẩn bị trước khi chạy:
1. Vào menu **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ Chọn **T4 GPU** $\rightarrow$ **Save**.
2. Chuẩn bị sẵn bộ dữ liệu đã gán nhãn (Export chuẩn **YOLOv8** từ Roboflow dưới dạng link API hoặc file `.zip`).

In [ ]:
# 1. Kiểm tra GPU & Cài đặt thư viện cần thiết
!nvidia-smi
!pip install -q ultralytics onnx onnxslim onnxruntime roboflow

---
## 📦 Bước 2: Nạp Bộ Dữ Liệu (Dataset)
Chọn **1 trong 2 cách** dưới đây để đưa dữ liệu đã gán nhãn vào Colab:

In [ ]:
# === CÁCH A (KHUYÊN DÙNG): Tải trực tiếp từ Roboflow ===
# Vào Roboflow -> Export Dataset -> Format: YOLOv8 -> Copy đoạn code 'show download code' dán vào dưới đây.

USE_ROBOFLOW = False  # Đổi thành True nếu dùng Roboflow API

if USE_ROBOFLOW:
    from roboflow import Roboflow
    rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
    project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
    version = project.version(1)
    dataset = version.download("yolov8")
    DATA_YAML_PATH = f"{dataset.location}/data.yaml"
else:
    # === CÁCH B: Upload thủ công file zip (datasets.zip) ===
    import zipfile
    from pathlib import Path
    from google.colab import files

    print("👉 Vui lòng chọn file .zip chứa dataset chuẩn YOLOv8 đã tải từ Roboflow/LabelImg...")
    uploaded = files.upload()
    zip_name = next(iter(uploaded.keys()))

    extract_dir = Path("/content/dataset")
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall(extract_dir)

    yaml_candidates = list(extract_dir.rglob("data.yaml"))
    assert yaml_candidates, "❌ Không tìm thấy file data.yaml bên trong file zip!"
    DATA_YAML_PATH = str(yaml_candidates[0])

print(f"✅ Đã sẵn sàng file cấu hình dataset: {DATA_YAML_PATH}")
!cat "{DATA_YAML_PATH}"

---
## 🧠 Bước 3: Huấn Luyện Mô Hình YOLOv8 Nano (Transfer Learning)

In [ ]:
from ultralytics import YOLO

# Nạp trọng số pretrained YOLOv8 Nano
model = YOLO("yolov8n.pt")

# Huấn luyện trên GPU
results = model.train(
    data=DATA_YAML_PATH,
    epochs=80,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    patience=20,
    project="/content/runs",
    name="smart_sorting_yolov8n",
    exist_ok=True,
    plots=True
)

---
## 📊 Bước 4: Đánh Giá Chất Lượng & Confusion Matrix

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run_dir = Path("/content/runs/smart_sorting_yolov8n")
best_pt = run_dir / "weights" / "best.pt"

# Đánh giá lại trên tập val
trained_model = YOLO(str(best_pt))
metrics = trained_model.val(data=DATA_YAML_PATH)
print(f"\n🎯 mAP@50: {metrics.box.map50:.4f} | mAP@50-95: {metrics.box.map:.4f}")

# Hiển thị biểu đồ huấn luyện & ma trận nhầm lẫn
for img_name in ["results.png", "confusion_matrix.png"]:
    img_path = run_dir / img_name
    if img_path.exists():
        print(f"\n--- {img_name} ---")
        display(Image(filename=str(img_path), width=800))

---
## ⚡ Bước 5: Xuất Mô Hình ONNX & Tải Về Máy Edge
Sau khi tải file `best.onnx` (và `best.pt`) về máy, hãy copy chúng vào thư mục `local_edge/models/` trong dự án của bạn.

In [ ]:
import numpy as np
import onnxruntime as ort
import shutil
from google.colab import files

# Export sang ONNX (opset=12 tương thích tốt với cả ONNX Runtime & OpenCV DNN)
onnx_path = trained_model.export(format="onnx", imgsz=640, simplify=True, opset=12, dynamic=False)

# Kiểm tra suy luận thử bằng ONNX Runtime
session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
in_name = session.get_inputs()[0].name
out = session.run(None, {in_name: np.random.rand(1, 3, 640, 640).astype(np.float32)})[0]
print(f"✅ Export ONNX thành công! File: {onnx_path} | Output shape: {out.shape}")

# Đóng gói hoặc tải trực tiếp 2 file trọng số về máy
shutil.copy2(best_pt, "/content/best.pt")
shutil.copy2(onnx_path, "/content/best.onnx")

files.download("/content/best.onnx")
files.download("/content/best.pt")